In [48]:
%pip install pandas
%pip install numpy
%pip install pyarrow
%pip install --upgrade pandas pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import pandas as pd
import numpy as np

file_path = '../data/techniques_classification/train.parquet'

df = pd.read_parquet(file_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3822 entries, 0 to 3821
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             3822 non-null   object
 1   content        3822 non-null   object
 2   lang           3822 non-null   object
 3   manipulative   3822 non-null   bool  
 4   techniques     2589 non-null   object
 5   trigger_words  2589 non-null   object
dtypes: bool(1), object(5)
memory usage: 153.2+ KB


In [50]:
df.head(30)

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"
5,46493f44-f00a-4ffb-9cda-252ccf5fa4c6,"Апартаменти\n триповерхова келія Паші Лєбєдя, ...",uk,True,[loaded_language],"[[94, 108], [208, 227]]"
6,1417fffa-461c-4bf7-bea5-398e8ea5db81,"⚡️\nКитай надеется, что пакетное соглашение о ...",ru,False,None,None
7,f8021666-1636-4e1f-b8e3-535860eca5ce,До сьогоднішнього дня Джо Байден був у Києві п...,uk,True,"[loaded_language, glittering_generalities, eup...","[[65, 223], [225, 697]]"
8,ed5d2195-09b4-4837-82eb-b65244c8a7b2,⚡️\nПосле освобождения Соледара откроются благ...,ru,True,"[cherry_picking, cliche]","[[3, 66], [71, 94]]"
9,7d7504ff-295e-435d-b565-ed62d4eebaa0,Российское руководство сделало террор инструме...,ru,True,"[loaded_language, cherry_picking, appeal_to_fear]","[[0, 80], [1034, 1049], [1076, 1105], [1171, 1..."


In [51]:
%pip install datasets transformers scikit-learn beautifulsoup4 pyarrow

import ast
import joblib
from bs4 import BeautifulSoup
from sklearn.preprocessing import MultiLabelBinarizer
from datasets import load_dataset, Dataset
import os


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
def parse_offsets(offsets_data):
    if offsets_data is None:
        return []
    
    if isinstance(offsets_data, (list, np.ndarray)):
        if len(offsets_data) == 0:
            return []
        return offsets_data

    try:
        if pd.isna(offsets_data):
            return []
    except ValueError:
        pass

    if isinstance(offsets_data, str):
        if not offsets_data.strip():
            return []
        try:
            return ast.literal_eval(offsets_data)
        except (ValueError, SyntaxError):
            return []
            
    return []


def clean_token_str(token):
    if re.match(r'http[s]?://\S+', token):
        return '[url]'
    
    cleaned = re.sub(r'[^\w\s,.:;!?@\'"()-]', '', token, flags=re.UNICODE)
    
    return cleaned.lower().strip()


def safe_eval_techniques(tech_str):
    if not tech_str or str(tech_str).lower() in ('none', 'nan', '[]'):
        return []
    
    s = str(tech_str)
    
    s = s.replace('[', ' ').replace(']', ' ').replace("'", ' ').replace('"', ' ').replace(',', ' ')
    tokens = s.split()
    
    return [t.strip().lower() for t in tokens if len(t) > 1 and not t.isdigit()]



In [53]:
import re
def process_row(row):
    
    raw_text = row['content']
    triggers_data = row['trigger_words']
    
    if not raw_text or pd.isna(raw_text):
        return "", []

    trigger_ranges = parse_offsets(triggers_data)
    
    final_tokens = []
    final_labels = []

    for match in re.finditer(r'\S+', raw_text):
        token_raw = match.group()
        start, end = match.span() 
        
        is_trigger = 0
        for (t_start, t_end) in trigger_ranges:
            if start < t_end and end > t_start:
                is_trigger = 1
                break
        
        token_clean = clean_token_str(token_raw)
        
        if token_clean:
            final_tokens.append(token_clean)
            final_labels.append(is_trigger)
            
    cleaned_full_text = " ".join(final_tokens)
    
    return pd.Series([cleaned_full_text, final_labels])

In [54]:
mlb_path = './multilabel_binarizer.pkl'

df[['cleaned_content', 'token_labels']] = df.apply(process_row, axis=1)
df['techniques'] = df['techniques'].fillna('').astype(str)
df['techniques_list'] = df['techniques'].apply(safe_eval_techniques)

mlb = MultiLabelBinarizer()
mlb.fit(df['techniques_list'])
binary_matrix = mlb.transform(df['techniques_list'])

techniques_df = pd.DataFrame(binary_matrix, columns=mlb.classes_)

joblib.dump(mlb, mlb_path)
print(f"Знайдено класів технік: {len(mlb.classes_)}")
print(f"Класи: {mlb.classes_}")


df_final = pd.concat([df[['id', 'cleaned_content', 'token_labels']], techniques_df], axis=1)

hf_dataset = Dataset.from_pandas(df_final)

sample_with_trigger = df_final[df_final['token_labels'].apply(lambda x: 1 in x if isinstance(x, list) else False)].head(1)




Знайдено класів технік: 10
Класи: ['appeal_to_fear' 'bandwagon' 'cherry_picking' 'cliche' 'euphoria' 'fud'
 'glittering_generalities' 'loaded_language' 'straw_man' 'whataboutism']


In [55]:
df_final.head()

,id,cleaned_content,token_labels,appeal_to_fear,bandwagon,cherry_picking,cliche,euphoria,fud,glittering_generalities,loaded_language,straw_man,whataboutism
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,новий огляд мапи deepstate від російського вій...,"[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,0,0,1,0,0,1,0,0
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,недавно 95 квартал жёстко поглумился над русск...,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0,1,0,0,0,0,1,0,0
2,e6a427f1-211f-405f-bd8b-70798458d656,тим часом йде евакуація бєлгородського автовок...,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, ...",0,0,0,0,1,0,0,1,0,0
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,в україні найближчим часом мають намір посилит...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0,0,0,0,0,0,0,0,0
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"расчёты 122-мм сау 2с1 ""гвоздика"" 132-й бригад...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0,0,0,0,0,0,1,0,0
